In [1]:
from tensorflow import keras
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(123)

In [2]:
def readucr(filename):
    data = np.loadtxt(filename, delimiter="\t")
    y = data[:, 0]
    x = data[:, 1:]
    return x, y.astype(int)

x_train, y_train = readucr("FordA_TRAIN.tsv")
x_test, y_test = readucr("FordA_TEST.tsv")

In [3]:
x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

print(x_train.shape)

(3601, 500, 1)


In [4]:
y_train[y_train==-1] = 0
y_test[y_test==-1] = 0

In [5]:
print(y_train[:10])

[0 1 0 0 0 1 1 1 1 1]


In [6]:
idx = np.random.permutation(len(x_train))
x_train = x_train[idx]
y_train = y_train[idx]

In [7]:
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)
train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

## Build a model

In [8]:
input_layer = keras.layers.Input((500, 1))

x = keras.layers.Conv1D(filters=64, kernel_size=3, padding="same")(input_layer)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)

x = keras.layers.Conv1D(filters=64, kernel_size=3, padding="same")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)

x = keras.layers.Conv1D(filters=64, kernel_size=3, padding="same")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)

x = keras.layers.GlobalAveragePooling1D()(x)
x = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.models.Model(inputs=input_layer, outputs=x)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 500, 1)]          0         
                                                                 
 conv1d (Conv1D)             (None, 500, 64)           256       
                                                                 
 batch_normalization (BatchN  (None, 500, 64)          256       
 ormalization)                                                   
                                                                 
 re_lu (ReLU)                (None, 500, 64)           0         
                                                                 
 conv1d_1 (Conv1D)           (None, 500, 64)           12352     
                                                                 
 batch_normalization_1 (Batc  (None, 500, 64)          256       
 hNormalization)                                             

## Train the model

In [9]:
epochs = 150
batch_size = 64

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])
history = model.fit(train_ds,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_data=(x_test, y_test),
                    verbose=2)

Epoch 1/150
113/113 - 14s - loss: 0.5546 - accuracy: 0.6948 - val_loss: 0.8302 - val_accuracy: 0.4841 - 14s/epoch - 121ms/step
Epoch 2/150
113/113 - 11s - loss: 0.4673 - accuracy: 0.7648 - val_loss: 0.8480 - val_accuracy: 0.4841 - 11s/epoch - 97ms/step
Epoch 3/150
113/113 - 11s - loss: 0.4365 - accuracy: 0.7870 - val_loss: 0.8437 - val_accuracy: 0.4841 - 11s/epoch - 97ms/step
Epoch 4/150
113/113 - 11s - loss: 0.4205 - accuracy: 0.7928 - val_loss: 0.6063 - val_accuracy: 0.5879 - 11s/epoch - 97ms/step
Epoch 5/150
113/113 - 11s - loss: 0.4101 - accuracy: 0.7989 - val_loss: 0.5565 - val_accuracy: 0.6409 - 11s/epoch - 98ms/step
Epoch 6/150
113/113 - 11s - loss: 0.3996 - accuracy: 0.8048 - val_loss: 0.4076 - val_accuracy: 0.8136 - 11s/epoch - 97ms/step
Epoch 7/150
113/113 - 11s - loss: 0.3913 - accuracy: 0.8117 - val_loss: 0.3853 - val_accuracy: 0.8265 - 11s/epoch - 97ms/step
Epoch 8/150
113/113 - 10s - loss: 0.3831 - accuracy: 0.8162 - val_loss: 0.6853 - val_accuracy: 0.6212 - 10s/epoch - 9

Epoch 66/150
113/113 - 11s - loss: 0.1144 - accuracy: 0.9611 - val_loss: 4.1992 - val_accuracy: 0.6591 - 11s/epoch - 93ms/step
Epoch 67/150
113/113 - 10s - loss: 0.1116 - accuracy: 0.9600 - val_loss: 2.8583 - val_accuracy: 0.6917 - 10s/epoch - 88ms/step
Epoch 68/150
113/113 - 10s - loss: 0.1088 - accuracy: 0.9625 - val_loss: 0.6197 - val_accuracy: 0.7636 - 10s/epoch - 87ms/step
Epoch 69/150
113/113 - 10s - loss: 0.1084 - accuracy: 0.9628 - val_loss: 0.1171 - val_accuracy: 0.9674 - 10s/epoch - 86ms/step
Epoch 70/150
113/113 - 10s - loss: 0.1113 - accuracy: 0.9628 - val_loss: 0.1086 - val_accuracy: 0.9682 - 10s/epoch - 93ms/step
Epoch 71/150
113/113 - 12s - loss: 0.1028 - accuracy: 0.9647 - val_loss: 0.1437 - val_accuracy: 0.9364 - 12s/epoch - 107ms/step
Epoch 72/150
113/113 - 11s - loss: 0.1037 - accuracy: 0.9631 - val_loss: 0.1406 - val_accuracy: 0.9576 - 11s/epoch - 100ms/step
Epoch 73/150
113/113 - 12s - loss: 0.1042 - accuracy: 0.9642 - val_loss: 0.4714 - val_accuracy: 0.8068 - 12s/

Epoch 131/150
113/113 - 8s - loss: 0.0887 - accuracy: 0.9703 - val_loss: 0.4196 - val_accuracy: 0.8121 - 8s/epoch - 71ms/step
Epoch 132/150
113/113 - 8s - loss: 0.0862 - accuracy: 0.9689 - val_loss: 0.1359 - val_accuracy: 0.9485 - 8s/epoch - 71ms/step
Epoch 133/150
113/113 - 8s - loss: 0.0818 - accuracy: 0.9714 - val_loss: 0.1707 - val_accuracy: 0.9250 - 8s/epoch - 71ms/step
Epoch 134/150
113/113 - 8s - loss: 0.0829 - accuracy: 0.9697 - val_loss: 0.8320 - val_accuracy: 0.7636 - 8s/epoch - 71ms/step
Epoch 135/150
113/113 - 8s - loss: 0.0791 - accuracy: 0.9747 - val_loss: 0.1468 - val_accuracy: 0.9409 - 8s/epoch - 71ms/step
Epoch 136/150
113/113 - 9s - loss: 0.0774 - accuracy: 0.9739 - val_loss: 3.1644 - val_accuracy: 0.7008 - 9s/epoch - 80ms/step
Epoch 137/150
113/113 - 9s - loss: 0.0759 - accuracy: 0.9731 - val_loss: 0.8827 - val_accuracy: 0.7439 - 9s/epoch - 84ms/step
Epoch 138/150
113/113 - 9s - loss: 0.0790 - accuracy: 0.9706 - val_loss: 0.1225 - val_accuracy: 0.9667 - 9s/epoch - 82

In [13]:
%matplotlib qt

metric = "accuracy"
plt.figure()
plt.plot(history.history[metric])
plt.plot(history.history["val_" + metric])
plt.title("model " + metric)
plt.ylabel(metric, fontsize="large")
plt.xlabel("epoch", fontsize="large")
plt.legend(["train", "val"], loc="best")
plt.show()